In [1]:
import re
import math
import random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

with open("corpus.txt", "r") as file:
    sents = to_sentences(file.read())

print("Sentences:", len(sents))

Sentences: 60


In [3]:
def build(sents, n):
    """Map each (n-1)-word context to a Counter of following words."""
    foll = defaultdict(Counter)

    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]

        for i in range(len(toks) - n + 1):
            context = tuple(toks[i:i + n - 1])
            next_word = toks[i + n - 1]
            foll[context][next_word] += 1

    return foll

In [4]:
bi = build(sents, 2)
tri = build(sents, 3)

print("Bigram contexts:", len(bi))
print("Trigram contexts:", len(tri))

Bigram contexts: 170
Trigram contexts: 352


In [5]:
def generate(foll, n, seed=0, max_len=25):
    rng = random.Random(seed)

    # Start with n-1 beginning tokens
    ctx = ("<s>",) * (n - 1)
    out = []

    for _ in range(max_len):
        c = foll[ctx]

        if not c:
            break

        words, weights = zip(*c.items())

        # Sample the next word using its frequency as weight
        nxt = rng.choices(words, weights=weights)[0]

        # Stop when the sentence-ending token is generated
        if nxt == "</s>":
            break

        out.append(nxt)

        # Keep only the most recent n-1 words as context
        ctx = (ctx + (nxt,))[-(n - 1):]

    return " ".join(out)

In [6]:
print("BIGRAM (1 word of memory):")

for s in range(4):
    print("  -", generate(bi, 2, seed=s))

print("\nTRIGRAM (2 words of memory):")

for s in range(4):
    print("  -", generate(tri, 3, seed=s))

BIGRAM (1 word of memory):
  - he enjoys a good book make strong
  - the piano softly on the park after a good book start my day
  - rainy mornings make me want more coffee and coffee
  - the quiet and coffee

TRIGRAM (2 words of memory):
  - he enjoys a long walk on sunny weekends
  - the mountains are beautiful in the green park
  - rainy mornings make me want more coffee
  - the garden is full of flowers in the park is quiet and full of old books


In [7]:
for seed in range(10):
    print(f"Seed {seed}: {generate(bi, 2, seed=seed)}")

Seed 0: he enjoys a good book make strong
Seed 1: the piano softly on the park after a good book start my day
Seed 2: rainy mornings make me want more coffee and coffee
Seed 3: the quiet and coffee
Seed 4: the city is quiet and a chapter of coffee in the station in the office is calm after a good book from the warm fire
Seed 5: she borrows a chapter of his coffee
Seed 6: he enjoys a good book from work when the corner makes the summer
Seed 7: the train was late again this morning
Seed 8: the early morning helps me wake up
Seed 9: the weekend we plan a warm fire in the dog runs happily in the summer


In [8]:
print("Short:")
print(generate(tri, 3, seed=5, max_len=5))

print("\nMedium:")
print(generate(tri, 3, seed=5, max_len=15))

print("\nLong:")
print(generate(tri, 3, seed=5, max_len=30))

Short:
she borrows a new book

Medium:
she borrows a new book from the library

Long:
she borrows a new book from the library


In [9]:
print("Bigram:")
print("Context:", ("<s>",))
print("Possible next words:", bi[("<s>",)].most_common())

print("\nTrigram:")
print("Context:", ("<s>", "<s>"))
print("Possible next words:", tri[("<s>", "<s>")].most_common())

Bigram:
Context: ('<s>',)
Possible next words: [('the', 23), ('i', 8), ('we', 8), ('she', 7), ('he', 7), ('morning', 2), ('on', 1), ('coffee', 1), ('rainy', 1), ('cold', 1), ('good', 1)]

Trigram:
Context: ('<s>', '<s>')
Possible next words: [('the', 23), ('i', 8), ('we', 8), ('she', 7), ('he', 7), ('morning', 2), ('on', 1), ('coffee', 1), ('rainy', 1), ('cold', 1), ('good', 1)]


## Observations — Lesson 4

1. Text generation is performed one word at a time using the
   previously generated context.

2. Weighted random sampling allows less frequent but valid
   next words to be selected.

3. The bigram model often produces less coherent text because
   it remembers only one previous word.

4. The trigram model generally produces more coherent local
   sequences because it uses two previous words.

5. Trigrams can reproduce phrases from the training corpus
   because the small dataset contains limited examples of
   many contexts.

6. Increasing the context length requires more training data.
   Otherwise, the model may encounter contexts with no
   recorded continuation.